# Historical Price Plots

For each round, this notebook loads every `prices_round_R_day_D.csv` file, concatenates the days in chronological order, and saves a mid-price chart per product into `assets/historical_prices/Round-R/`.

In [1]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "assets").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repo root (expected data/ and assets/)")


ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "assets" / "historical_prices"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMPS_PER_DAY = 1_000_000  # iTimestamp resolution per Prosperity day

In [2]:
def load_round(round_dir: Path) -> pd.DataFrame:
    """Load all price CSVs for a round, sorted by day, with a continuous time axis."""
    files = sorted(
        round_dir.glob("prices_round_*_day_*.csv"),
        key=lambda p: int(re.search(r"day_(-?\d+)", p.stem).group(1)),
    )
    frames = []
    days_seen = []
    for f in files:
        df = pd.read_csv(f, sep=";")
        days_seen.append(int(df["day"].iloc[0]))
        frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    # Make a global time axis spanning all days in order.
    day_order = {d: i for i, d in enumerate(sorted(set(days_seen)))}
    combined["global_t"] = (
        combined["day"].map(day_order) * TIMESTAMPS_PER_DAY + combined["timestamp"]
    )
    return combined.sort_values(["product", "global_t"]).reset_index(drop=True)

In [3]:
def plot_round(round_dir: Path) -> None:
    round_name = round_dir.name  # e.g. "Round-1"
    out = OUT_DIR / round_name
    out.mkdir(parents=True, exist_ok=True)

    df = load_round(round_dir)
    days = sorted(df["day"].unique())
    print(f"{round_name}: {len(df['product'].unique())} products, days {days}")

    for product, sub in df.groupby("product"):
        sub = sub.copy()
        # Recompute mid from best bid/ask. mid_price=0 in the source means an empty
        # book at that timestamp; drop those so they don't show as spikes to zero.
        both = sub["bid_price_1"].notna() & sub["ask_price_1"].notna()
        sub.loc[both, "mid_price"] = (sub.loc[both, "bid_price_1"] + sub.loc[both, "ask_price_1"]) / 2
        one_side = ~both
        sub.loc[one_side, "mid_price"] = sub.loc[one_side, "bid_price_1"].fillna(
            sub.loc[one_side, "ask_price_1"]
        )
        sub = sub.dropna(subset=["mid_price"])
        sub = sub[sub["mid_price"] > 0]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(sub["global_t"], sub["mid_price"], linewidth=0.8, color="#1f77b4")

        # Day boundaries and labels.
        for i, d in enumerate(sorted(sub["day"].unique())):
            if i > 0:
                ax.axvline(i * TIMESTAMPS_PER_DAY, color="grey", alpha=0.3, linestyle="--")
        ax.set_xticks([(i + 0.5) * TIMESTAMPS_PER_DAY for i in range(len(days))])
        ax.set_xticklabels([f"day {d}" for d in days])

        ax.set_title(f"{round_name} — {product} mid price")
        ax.set_xlabel("time")
        ax.set_ylabel("mid price")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(out / f"{product}.png", dpi=120)
        plt.close(fig)
    print(f"  → saved to {out.relative_to(ROOT)}")

In [4]:
round_dirs = sorted(DATA_DIR.glob("Round-*"), key=lambda p: int(p.name.split("-")[1]))
for rd in round_dirs:
    plot_round(rd)
print("done")

Round-1: 2 products, days [np.int64(-2), np.int64(-1), np.int64(0)]
  → saved to assets/historical_prices/Round-1


Round-2: 2 products, days [np.int64(-1), np.int64(0), np.int64(1)]
  → saved to assets/historical_prices/Round-2


Round-3: 12 products, days [np.int64(0), np.int64(1), np.int64(2)]


  → saved to assets/historical_prices/Round-3


Round-4: 12 products, days [np.int64(1), np.int64(2), np.int64(3)]


  → saved to assets/historical_prices/Round-4


Round-5: 50 products, days [np.int64(2), np.int64(3), np.int64(4)]


  → saved to assets/historical_prices/Round-5
done
